## Import

In [1]:
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
print("=" * 60)
print("BATCH SCORING — Ensemble Fraud Tier Assignment")
print("=" * 60)
 


StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 3, Finished, Available, Finished, False)

BATCH SCORING — Ensemble Fraud Tier Assignment


##  Load Both Registered Models 

In [2]:
try:
    xgb_model = mlflow.sklearn.load_model("models:/FraudXGBoost/Production")
    print("XGBoost loaded: Production stage")
except Exception:
    xgb_model = mlflow.sklearn.load_model("models:/FraudXGBoost/1")
    print("XGBoost loaded: Version 1")
 
try:
    iso_model = mlflow.sklearn.load_model("models:/FraudIsolationForest/Production")
    print("IsolationForest loaded: Production stage")
except Exception:
    iso_model = mlflow.sklearn.load_model("models:/FraudIsolationForest/1")
    print("IsolationForest loaded: Version 1")

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 4, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


IsolationForest loaded: Version 1


## Load Features

In [3]:
FEAT_COLS = [f"V{i}" for i in range(1, 29)] + ["Amount", "hour_of_day"]
 
df_spark = spark.read.format("delta").table("silver_features")
df_pd = df_spark.select(["Time", "Amount", "Class", "hour_of_day", "amount_bin"] +\
 [f"V{i}" for i in range(1, 29)]).toPandas()
 
print(f"\nTransactions to score: {len(df_pd):,}")
X = df_pd[FEAT_COLS].fillna(0)

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 5, Finished, Available, Finished, False)


Transactions to score: 284,807


In [4]:
display(df_pd)

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9db67329-ee9b-457c-8d45-165d1b4c9e78)

##  XGBoost Fraud Probability

In [5]:
df_pd["fraud_prob"]      = xgb_model.predict_proba(X)[:, 1]
df_pd["fraud_score_pct"] = (df_pd["fraud_prob"] * 100).round(1)
 
# ── CELL 5: Isolation Forest Anomaly Score ────────────────
raw_iso_scores    = iso_model.decision_function(X)
df_pd["iso_score"] = raw_iso_scores
df_pd["iso_flag"]  = (iso_model.predict(X) == -1).astype(int)
 
# Normalise IF score to 0–100 for display in Power BI
iso_min, iso_max = raw_iso_scores.min(), raw_iso_scores.max()
df_pd["iso_score_pct"] = (
    (1 - (raw_iso_scores - iso_min) / (iso_max - iso_min)) * 100
).round(1)

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 7, Finished, Available, Finished, False)

In [6]:
display(df_pd)

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aac46f01-d372-4e8e-982d-bf3ce9871cbd)

## Ensemble Fraud Tier

In [7]:
# CRITICAL: XGBoost says high prob OR Isolation Forest flags anomaly
#           → Both models agree something is wrong
# HIGH:     XGBoost says moderate probability
#           → One signal, needs review
# MONITOR:  Low probability, no anomaly flag
#           → Normal transaction
#
# The ensemble catches fraud that each model misses alone:
# - New fraud patterns (unseen by XGBoost) → IF catches them
# - Patterns matching known fraud → XGBoost catches them
 
# df_pd["fraud_tier"] = np.where(
#     (df_pd["fraud_prob"] >= 0.80) | (df_pd["iso_flag"] == 1),
#     "CRITICAL",
#     np.where(df_pd["fraud_prob"] >= 0.50, "HIGH", "MONITOR")
# )
 
# # Urgency sub-tier for Data Activator action routing
# df_pd["action_required"] = np.where(
#     (df_pd["fraud_score_pct"] >= 90) & (df_pd["Amount"] >= 500),
#     "BLOCK_NOW",
#     np.where(
#         (df_pd["fraud_score_pct"] >= 80) | (df_pd["iso_flag"] == 1),
#         "FLAG_REVIEW",
#         "MONITOR"
#     )
# )

# ── REVISED TIER LOGIC ────────────────────────────────────
# Remove HIGH tier — no fraud scores in 50-80% range
# Use 0.70 threshold based on threshold sensitivity analysis
# Keep iso_flag for production streaming (future-proofing)

# df_pd["fraud_tier"] = np.where(
#     (df_pd["fraud_prob"] >= 0.70) | (df_pd["iso_flag"] == 1),
#     "CRITICAL",
#     "MONITOR"
# )

# # Urgency sub-tier for Data Activator action routing
# df_pd["action_required"] = np.where(
#     (df_pd["fraud_score_pct"] >= 90) & (df_pd["Amount"] >= 500),
#     "BLOCK_NOW",       # highest confidence + high value: auto-block
#     np.where(
#         df_pd["fraud_tier"] == "CRITICAL",
#         "FLAG_REVIEW", # review queue for fraud analyst
#         "MONITOR"
#     )
# )

# ── REVISED: XGBoost only for batch CRITICAL tier ─────────
# iso_flag removed from batch scoring — adds 0 unique fraud
# and contributes 569 false positives to the CRITICAL tier.
# Isolation Forest is retained in the architecture ONLY for
# the Eventstream real-time layer where novel fraud patterns
# may appear that XGBoost has not learned.

df_pd["fraud_tier"] = np.where(
    df_pd["fraud_prob"] >= 0.70,   # XGBoost only
    "CRITICAL",
    "MONITOR"
)

df_pd["action_required"] = np.where(
    (df_pd["fraud_score_pct"] >= 90) & (df_pd["Amount"] >= 500),
    "BLOCK_NOW",
    np.where(df_pd["fraud_tier"] == "CRITICAL", "FLAG_REVIEW", "MONITOR")
)

# Keep iso_flag and iso_score as stored columns in silver_fraud_scores
# for the real-time Eventstream use case — just don't use them
# to determine the batch CRITICAL tier.

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 9, Finished, Available, Finished, False)

In [8]:
# ── Verify Revised Tier Logic & Action Routing ──

# Baseline Totals
total_txns = len(df_pd)
total_fraud = df_pd["Class"].sum()

# 1. CRITICAL Tier Stats
crit_mask = df_pd["fraud_tier"] == "CRITICAL"
crit_total = crit_mask.sum()
crit_caught = df_pd.loc[crit_mask, "Class"].sum()
crit_fp = crit_total - crit_caught
crit_precision = (crit_caught / crit_total) * 100 if crit_total > 0 else 0

crit_txn_pct = (crit_total / total_txns) * 100 if total_txns > 0 else 0
crit_fraud_pct = (crit_caught / total_fraud) * 100 if total_fraud > 0 else 0

# 2. Action Routing Stats (Within CRITICAL)
block_now_count = (df_pd["action_required"] == "BLOCK_NOW").sum()
flag_review_count = (df_pd["action_required"] == "FLAG_REVIEW").sum()

# 3. MONITOR Tier Stats
mon_mask = df_pd["fraud_tier"] == "MONITOR"
mon_total = mon_mask.sum()
mon_missed = df_pd.loc[mon_mask, "Class"].sum()  # Fraud that slipped through
mon_tn = mon_total - mon_missed                  # Legitimate txns correctly ignored

mon_txn_pct = (mon_total / total_txns) * 100 if total_txns > 0 else 0
mon_fraud_pct = (mon_missed / total_fraud) * 100 if total_fraud > 0 else 0

# 4. Print the Verification Summary
print(f"\nAfter Fix — Revised Performance Summary:")
print("=" * 75)
print(f"CRITICAL : {crit_total:>7,} txns ({crit_txn_pct:>5.2f}%) | fraud caught: {crit_caught:>4,} ({crit_fraud_pct:.0f}%)")
print(f"  Precision @ CRITICAL : {crit_precision:.2f}%")
print(f"  False Positives      : {crit_fp:,} (False alarms)\n")

print(f"Action Routing (Within CRITICAL):")
print(f"  BLOCK_NOW (Auto)     : {block_now_count:>7,} txns")
print(f"  FLAG_REVIEW (Human)  : {flag_review_count:>7,} txns\n")

print("-" * 75)
print(f"MONITOR  : {mon_total:>7,} txns ({mon_txn_pct:>5.2f}%) | fraud caught: {mon_missed:>4,} ({mon_fraud_pct:.0f}%)")
print(f"  True Negatives       : {mon_tn:,} (Legit txns ignored)")
print("=" * 75)

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 10, Finished, Available, Finished, False)


After Fix — Revised Performance Summary:
CRITICAL :     506 txns ( 0.18%) | fraud caught:  478 (97%)
  Precision @ CRITICAL : 94.47%
  False Positives      : 28 (False alarms)

Action Routing (Within CRITICAL):
  BLOCK_NOW (Auto)     :      33 txns
  FLAG_REVIEW (Human)  :     473 txns

---------------------------------------------------------------------------
MONITOR  : 284,301 txns (99.82%) | fraud caught:   14 (3%)
  True Negatives       : 284,287 (Legit txns ignored)


## Performance Summary

In [9]:
total_fraud = df_pd["Class"].sum()
print(f"\nTotal actual fraud transactions : {total_fraud:,}")
print(f"\nEnsemble tier distribution:")
for tier in ["CRITICAL", "HIGH", "MONITOR"]:
    subset  = df_pd[df_pd.fraud_tier == tier]
    cnt     = len(subset)
    caught  = subset["Class"].sum()
    pct_all = cnt / len(df_pd) * 100
    print(f"  {tier:<10}: {cnt:>7,} txns ({pct_all:.2f}%) | "
          f"fraud caught: {caught:>4,} ({caught/total_fraud*100:.0f}%)")
 
print(f"\nEnsemble capture analysis:")
critical_caught = df_pd[df_pd.fraud_tier=="CRITICAL"]["Class"].sum()
high_caught     = df_pd[df_pd.fraud_tier.isin(["CRITICAL","HIGH"])]["Class"].sum()
print(f"  CRITICAL tier captures         : {critical_caught/total_fraud:.0%} of all fraud")
print(f"  CRITICAL+HIGH captures         : {high_caught/total_fraud:.0%} of all fraud")
print(f"  CRITICAL tier precision        : "
      f"{critical_caught/len(df_pd[df_pd.fraud_tier=='CRITICAL']):.2%}")
 
# Compare to single-model
xgb_caught = df_pd[df_pd.fraud_prob >= 0.80]["Class"].sum()
iso_caught  = df_pd[df_pd.iso_flag == 1]["Class"].sum()
ens_caught  = df_pd[
    (df_pd.fraud_prob >= 0.80) | (df_pd.iso_flag == 1)
]["Class"].sum()
print(f"\nSingle vs ensemble comparison:")
print(f"  XGBoost alone (≥70%)   : {xgb_caught:,} fraud caught")
print(f"  Isolation Forest alone : {iso_caught:,} fraud caught")
print(f"  Ensemble (either)      : {ens_caught:,} fraud caught  ← improvement")

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 11, Finished, Available, Finished, False)


Total actual fraud transactions : 492

Ensemble tier distribution:
  CRITICAL  :     506 txns (0.18%) | fraud caught:  478 (97%)
  HIGH      :       0 txns (0.00%) | fraud caught:    0 (0%)
  MONITOR   : 284,301 txns (99.82%) | fraud caught:   14 (3%)

Ensemble capture analysis:
  CRITICAL tier captures         : 97% of all fraud
  CRITICAL+HIGH captures         : 97% of all fraud
  CRITICAL tier precision        : 94.47%

Single vs ensemble comparison:
  XGBoost alone (≥70%)   : 478 fraud caught
  Isolation Forest alone : 133 fraud caught
  Ensemble (either)      : 478 fraud caught  ← improvement


## Write Fraud Scores

In [11]:
df_out = df_pd[[
    "Time", "Amount", "hour_of_day", "amount_bin",
    "fraud_score_pct","iso_score","iso_score_pct", "iso_flag",
    "fraud_tier","fraud_prob", "action_required", "Class"
]]
spark.createDataFrame(df_out) \
     .write.format("delta").mode("overwrite") \
     .option("overwriteSchema", "true") \
     .saveAsTable("silver_fraud_scores")
 
print(f"\n✅ silver_fraud_scores: {len(df_out):,} rows")
 
# Score distribution
spark.read.format("delta").table("silver_fraud_scores") \
     .select(
         F.min("fraud_score_pct").alias("min"),
         F.avg("fraud_score_pct").alias("mean"),
         F.percentile_approx("fraud_score_pct", 0.5).alias("median"),
         F.percentile_approx("fraud_score_pct", 0.99).alias("p99"),
         F.max("fraud_score_pct").alias("max")
     ).show()

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 13, Finished, Available, Finished, False)


✅ silver_fraud_scores: 284,807 rows
+---+-------------------+------+---+-----+
|min|               mean|median|p99|  max|
+---+-------------------+------+---+-----+
|0.0|0.24257128575555234|   0.0|1.3|100.0|
+---+-------------------+------+---+-----+



In [12]:
# Detailed Overlap Analysis
# After both models have scored df_pd

# Filter for actual fraud (Class==1) and get the index for set operations
xgb_caught = set(df_pd[(df_pd["fraud_prob"] >= 0.70) & (df_pd["Class"] == 1)].index)
iso_caught = set(df_pd[(df_pd["iso_flag"] == 1) & (df_pd["Class"] == 1)].index)

only_xgb = xgb_caught - iso_caught
only_iso = iso_caught - xgb_caught
both     = xgb_caught & iso_caught

print(f"\nDetailed Overlap Breakdown:")
print(f"Caught by XGBoost only    : {len(only_xgb)}")
print(f"Caught by IF only         : {len(only_iso)}   # the ensemble value") 
print(f"Caught by both            : {len(both)}")
print(f"Total unique caught       : {len(xgb_caught | iso_caught)}")
print(f"Total missed by both      : {492 - len(xgb_caught | iso_caught)}")

StatementMeta(, d296df12-71a8-4f10-bcf3-ca7b80507810, 14, Finished, Available, Finished, False)


Detailed Overlap Breakdown:
Caught by XGBoost only    : 345
Caught by IF only         : 0   # the ensemble value
Caught by both            : 133
Total unique caught       : 478
Total missed by both      : 14
